In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName("Case Study") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/16 04:20:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark = SparkSession.builder \
    .appName("Case Study") \
    .getOrCreate()

26/06/15 04:47:21 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [9]:
customers = spark.read.csv("data/customers.csv", header=True, inferSchema=True)
products = spark.read.csv("data/products.csv", header=True, inferSchema=True)
orders = spark.read.csv("data/orders.csv", header=True, inferSchema=True)
returns = spark.read.csv("data/returns.csv", header=True, inferSchema=True)


In [10]:
total_customers = customers.count()
total_products = products.count()
total_orders = orders.count()
total_returned_orders = returns.count()

In [11]:
print(f"Total Customers: {total_customers}")
print(f"Total Products: {total_products}")
print(f"Total Orders: {total_orders}")
print(f"Total Returned Orders: {total_returned_orders}")

Total Customers: 100000
Total Products: 50000
Total Orders: 1000000
Total Returned Orders: 100000


In [12]:
from pyspark.sql.functions import sum, col
order_items = spark.read.csv("data/order_items.csv", header=True, inferSchema=True)
sales_data = order_items.join(products, "product_id")

sales_data.groupBy("category") \
    .agg(
        sum(col("selling_price") * col("quantity"))
        .alias("total_sales")
    ) \
    .show()

[Stage 43:>                                                         (0 + 2) / 2]

+--------------+-------------------+
|      category|        total_sales|
+--------------+-------------------+
|Home & Kitchen|7.581388732799902E8|
|        Sports|7.433388681300008E8|
|   Electronics|7.442665041099958E8|
|      Clothing|7.419227945699946E8|
|         Books|7.464907783499908E8|
|        Beauty|7.626693058999963E8|
|          Toys|7.446190722999846E8|
+--------------+-------------------+



In [16]:
from pyspark.sql.functions import sum, col

top_customers = orders.join(order_items, "order_id") \
    .groupBy("customer_id") \
    .agg(
        sum(col("selling_price") * col("quantity"))
        .alias("total_purchase")
    ) \
    .orderBy(col("total_purchase").desc()) \
    .limit(10)

top_customers.show()

[Stage 53:>                                                         (0 + 2) / 2]

+-----------+------------------+
|customer_id|    total_purchase|
+-----------+------------------+
|      93094|         181569.68|
|      64560|169060.39999999997|
|      23289|161573.80000000002|
|      52275|153364.78999999998|
|      61218|         153067.55|
|      52034|         152680.05|
|      40442|         151037.32|
|      60528|         148691.95|
|      84830|         148363.84|
|      82593|148281.03999999998|
+-----------+------------------+



In [17]:
from pyspark.sql.functions import year, month, max, sum, col

In [18]:
last_year = orders.select(
    max(year("order_date"))
).collect()[0][0]

print(last_year)

2024


In [19]:
orders_last_year = orders.filter(
    year("order_date") == last_year
)

In [20]:
sales_df = orders_last_year.join(
    order_items,
    "order_id"
)

In [21]:
monthly_sales = sales_df.groupBy(
    month("order_date").alias("month")
).agg(
    sum(
        col("selling_price") *
        col("quantity")
    ).alias("total_sales")
).orderBy("month")

monthly_sales.show()

[Stage 65:>                                                         (0 + 2) / 2]

+-----+--------------------+
|month|         total_sales|
+-----+--------------------+
|    1| 4.445777757600032E8|
|    2| 4.153661441999996E8|
|    3| 4.436282454099993E8|
|    4|4.2782097433999574E8|
|    5|4.4481061894999886E8|
|    6| 4.317051540600023E8|
|    7| 4.436705191199994E8|
|    8|4.4109517702000153E8|
|    9| 4.310715260799986E8|
|   10|4.4136378931000113E8|
|   11| 4.336233640400014E8|
|   12| 4.427129083499967E8|
+-----+--------------------+



In [22]:
df = order_items.join(
    products,
    "product_id"
).join(
    orders,
    "order_id"
)

In [23]:
from pyspark.sql.functions import year, month, max, sum, col

In [13]:
print(orders)
print(products)

DataFrame[order_id: int, customer_id: int, order_date: date, payment_mode: string, order_status: string]
DataFrame[product_id: int, product_name: string, category: string, brand: string, unit_cost: double]


In [14]:
order_items = spark.read.csv(
    "data/order_items.csv",
    header=True,
    inferSchema=True
)


In [15]:
order_items.show(5)

+-------------+--------+----------+--------+-------------+
|order_item_id|order_id|product_id|quantity|selling_price|
+-------------+--------+----------+--------+-------------+
|            1|  227444|     28849|       5|       727.98|
|            2|   32708|     25471|       2|       203.02|
|            3|  426242|      2818|       5|      1061.81|
|            4|  236965|     35931|       5|      1005.49|
|            5|  289552|     48310|       4|       540.78|
+-------------+--------+----------+--------+-------------+
only showing top 5 rows



In [26]:
total_orders.show()

[Stage 72:=============================>                            (1 + 1) / 2]

+--------------+------+
|      category| count|
+--------------+------+
|Home & Kitchen|434034|
|        Sports|424412|
|   Electronics|425896|
|      Clothing|427607|
|         Books|427086|
|        Beauty|430547|
|          Toys|430418|
+--------------+------+



In [27]:
returned_orders.show()

[Stage 77:=============================>                            (1 + 1) / 2]

+--------------+-----+
|      category|count|
+--------------+-----+
|Home & Kitchen|72468|
|        Sports|70571|
|   Electronics|70615|
|      Clothing|71106|
|         Books|71090|
|        Beauty|71860|
|          Toys|72387|
+--------------+-----+



In [29]:
from pyspark.sql.functions import col

total_orders = df.groupBy("category") \
    .count() \
    .withColumnRenamed("count", "total_orders")

returned_orders = df.filter(
    col("order_status") == "Returned"
).groupBy("category") \
 .count() \
 .withColumnRenamed("count", "returned_orders")

In [30]:
return_percentage = returned_orders.join(
    total_orders,
    "category"
).withColumn(
    "return_percentage",
    (col("returned_orders") / col("total_orders")) * 100
)

return_percentage.show()

[Stage 84:>                 (0 + 2) / 2][Stage 86:>                 (0 + 0) / 1]

+--------------+---------------+------------+------------------+
|      category|returned_orders|total_orders| return_percentage|
+--------------+---------------+------------+------------------+
|Home & Kitchen|          72468|      434034|16.696387840583917|
|        Sports|          70571|      424412| 16.62794642941293|
|   Electronics|          70615|      425896|16.580338862069613|
|      Clothing|          71106|      427607| 16.62882038881496|
|         Books|          71090|      427086|16.645359482633474|
|        Beauty|          71860|      430547|16.690396170452935|
|          Toys|          72387|      430418|16.817837543968885|
+--------------+---------------+------------+------------------+



In [32]:
payment_count = df.groupBy(
    "state",
    "payment_mode"
).count()

payment_count.show()

[Stage 97:>                                                         (0 + 2) / 2]

+-----+----------------+-----+
|state|    payment_mode|count|
+-----+----------------+-----+
|   FL|     Net Banking|19629|
|   NC|     Net Banking|19596|
|   GA|      Debit Card|19733|
|   OH|             UPI|19930|
|   MI|Cash on Delivery|20205|
|   IL|Cash on Delivery|20498|
|   OH|     Net Banking|20351|
|   TX|     Credit Card|19874|
|   IL|             UPI|20359|
|   NY|             UPI|20108|
|   NY|     Net Banking|20229|
|   TX|             UPI|20065|
|   GA|     Credit Card|19722|
|   TX|      Debit Card|19988|
|   MI|     Net Banking|20116|
|   NY|Cash on Delivery|20208|
|   CA|      Debit Card|20024|
|   IL|     Net Banking|20404|
|   CA|     Credit Card|19979|
|   OH|     Credit Card|19794|
+-----+----------------+-----+
only showing top 20 rows



In [33]:
df = orders.join(
    customers,
    "customer_id"
)

In [39]:
payment_count = df.groupBy(
    "state",
    "payment_mode"
).count()

payment_count.show()

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `state` cannot be resolved. Did you mean one of the following? [`brand`, `category`, `quantity`, `order_id`, `unit_cost`].;
'Aggregate ['state, 'payment_mode], ['state, 'payment_mode, count(1) AS count#1090L]
+- Project [product_id#385, order_item_id#383, order_id#384, quantity#386, selling_price#387, product_name#195, category#196, brand#197, unit_cost#198]
   +- Join Inner, (product_id#385 = product_id#194)
      :- Relation [order_item_id#383,order_id#384,product_id#385,quantity#386,selling_price#387] csv
      +- Relation [product_id#194,product_name#195,category#196,brand#197,unit_cost#198] csv


In [35]:
payment_count.orderBy(
    "state",
    col("count").desc()
).show()

[Stage 105:>                                                        (0 + 2) / 2]

+-----+----------------+-----+
|state|    payment_mode|count|
+-----+----------------+-----+
|   CA|             UPI|20246|
|   CA|     Net Banking|20054|
|   CA|      Debit Card|20024|
|   CA|Cash on Delivery|19999|
|   CA|     Credit Card|19979|
|   FL|      Debit Card|20010|
|   FL|     Credit Card|19888|
|   FL|Cash on Delivery|19762|
|   FL|             UPI|19740|
|   FL|     Net Banking|19629|
|   GA|     Net Banking|20041|
|   GA|Cash on Delivery|19771|
|   GA|      Debit Card|19733|
|   GA|     Credit Card|19722|
|   GA|             UPI|19608|
|   IL|Cash on Delivery|20498|
|   IL|     Net Banking|20404|
|   IL|     Credit Card|20361|
|   IL|             UPI|20359|
|   IL|      Debit Card|20178|
+-----+----------------+-----+
only showing top 20 rows



In [36]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("state") \
    .orderBy(col("count").desc())

final_answer = payment_count.withColumn(
    "rank",
    row_number().over(window_spec)
).filter(
    col("rank") == 1
)

final_answer.show()

[Stage 109:>                                                        (0 + 2) / 2]

+-----+----------------+-----+----+
|state|    payment_mode|count|rank|
+-----+----------------+-----+----+
|   CA|             UPI|20246|   1|
|   FL|      Debit Card|20010|   1|
|   GA|     Net Banking|20041|   1|
|   IL|Cash on Delivery|20498|   1|
|   MI|      Debit Card|20416|   1|
|   NC|     Net Banking|19596|   1|
|   NY|      Debit Card|20369|   1|
|   OH|     Net Banking|20351|   1|
|   TX|             UPI|20065|   1|
|   WA|             UPI|20244|   1|
+-----+----------------+-----+----+



In [37]:
from pyspark.sql.functions import (
    col,
    sum,
    countDistinct
)

df = orders.join(
    order_items,
    "order_id"
).join(
    products,
    "product_id"
)

result = df.groupBy(
    "customer_id"
).agg(
    countDistinct("category")
    .alias("category_count"),

    sum(
        col("selling_price") *
        col("quantity")
    ).alias("total_spent")
)

qualified_customers = result.filter(
    (col("category_count") >= 5) &
    (col("total_spent") > 100000)
)

qualified_customers.show()

26/06/16 04:49:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 04:49:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
[Stage 124:>                                                        (0 + 2) / 2]

+-----------+--------------+------------------+
|customer_id|category_count|       total_spent|
+-----------+--------------+------------------+
|      82672|             7|         104905.22|
|      64423|             7|122832.84000000001|
|      91446|             7|         122214.13|
|      91367|             7|          107376.9|
|      57984|             7|127489.03999999998|
|       9376|             7|107470.09999999999|
|      85749|             7|         109065.03|
|      81900|             7|         127465.68|
|      85100|             7|101890.51999999999|
|       2659|             7|         116119.88|
|      60965|             7|         104585.06|
|      22990|             7|115511.68999999999|
|      43588|             7|111164.06999999999|
|      68433|             7|108316.26999999999|
|      80943|             7|118172.32999999999|
|      68544|             7|         115948.79|
|      35450|             7|109224.42000000001|
|      36655|             7|         104

In [38]:
from pyspark.sql.functions import sum, col, row_number
from pyspark.sql.window import Window


df = order_items.join(
    products,
    "product_id"
)


product_revenue = df.groupBy(
    "category",
    "product_name"
).agg(
    sum(
        col("selling_price") * col("quantity")
    ).alias("revenue")
)


window_spec = Window.partitionBy(
    "category"
).orderBy(
    col("revenue").desc()
)


top_3_products = product_revenue.withColumn(
    "rank",
    row_number().over(window_spec)
).filter(
    col("rank") <= 3
)

top_3_products.show()

[Stage 131:============================>                            (1 + 1) / 2]

+--------------+-------------+------------------+----+
|      category| product_name|           revenue|rank|
+--------------+-------------+------------------+----+
|        Beauty|Product_44016|         277567.99|   1|
|        Beauty|Product_14849|274894.19999999995|   2|
|        Beauty|  Product_786|272174.69999999995|   3|
|         Books|Product_35314|         296468.78|   1|
|         Books|Product_28311|286757.72000000003|   2|
|         Books|Product_37479|276736.70999999996|   3|
|      Clothing| Product_7025| 293821.9699999999|   1|
|      Clothing| Product_1560| 288474.0900000001|   2|
|      Clothing|Product_31322| 282241.1699999999|   3|
|   Electronics| Product_6719|         299113.87|   1|
|   Electronics|Product_23519|         289561.72|   2|
|   Electronics|Product_38170|         288875.23|   3|
|Home & Kitchen| Product_5012| 305836.2200000001|   1|
|Home & Kitchen|Product_37452| 286817.4199999999|   2|
|Home & Kitchen|Product_27682|283340.36000000004|   3|
|        S

In [ ]:
from pyspark.sql.function